# 価値反復法

価値反復法は、方策を明示的に評価せず、価値の最適ベルマン更新を直接反復します。状態数が大きい問題で計算を単純化しやすいのが利点です。


## 参考動画
- [対応動画 1](https://www.youtube.com/watch?v=BYh4uwRgNnk)
@[youtube](https://www.youtube.com/watch?v=BYh4uwRgNnk)

## 参考リンク
- https://www.youtube.com/watch?v=BYh4uwRgNnk


## 背景と目的

方策反復法との違いは、毎回の改善ステップを省いて価値更新に集約する点です。

両者の更新式の違いを理解すると、問題規模に応じた手法選択ができます。

同じ2状態MDPで価値反復を実行し、最終方策を抽出します。


## 最初に解きたい疑問

1. 方策反復と比べて、どの手順を省いているのか。
2. `max(action_value(...))` は、何を最適化しているのか。
3. `diff` が小さくなったら止めてよい理由は何か。
4. `policy_star` は、どこから復元しているのか。
5. 価値が収束したあとに方策を作る順番は、なぜ必要なのか。


## 先に押さえる言葉

- `value iteration`: 価値の最適更新を繰り返して、最終的に最適方策を得る方法です。方策評価を明示的に分けません。
- `optimal value function`: 最善の行動を選んだときに得られる価値です。方策の制約を外した上限の価値になります。
- `convergence threshold`: 更新前後の差がこの値より小さくなったら止める基準です。計算を打ち切る実務上の目安になります。
- `greedy policy`: 得られた価値に対して、各状態で最も価値の高い行動だけを選ぶ方策です。


## 実行前の見取り図

1. 途中の `iter=... diff=...` が、収束に向かって小さくなっているか。
2. `optimal V = ...` が、反復後の価値として出ているか。
3. `greedy policy from V = ...` で、価値から方策が復元できているか。


## つまずきやすい点

- `diff` を停止条件にする意味を、最適化誤差との関係で説明する補足がない。
- 方策反復との違いを、ループ構造の差として見せる説明が足りない。


## コード 1: 更新式や補助関数を定義する

このセルでは 更新式や補助関数を定義する ための最小コードを動かします。 実行時は「途中の `iter=... diff=...` が、収束に向かって小さくなっているか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
states = ['s0', 's1']
actions = ['left', 'right']
gamma = 0.9

P = {
    's0': {
        'left': [('s0', 0.2), ('s1', 1.0)],
        'right': [('s1', 0.4), ('s1', 0.4)],
    },
    's1': {
        'left': [('s0', 0.0), ('s0', 0.0)],
        'right': [('s1', 0.8), ('s1', 0.8)],
    },
}

def action_value(s, a, V):
    trans = P[s][a]
    return sum(0.5 * (r + gamma * V[ns]) for ns, r in trans)


## コード 2: 価値反復を回して方策を取り出す

このセルでは 価値反復を回して方策を取り出す ための最小コードを動かします。 実行時は「`optimal V = ...` が、反復後の価値として出ているか。」を意識して、変数名と出力の対応を追ってください。


In [ ]:
V = {'s0': 0.0, 's1': 0.0}

for it in range(40):
    newV = {}
    for s in states:
        newV[s] = max(action_value(s, a, V) for a in actions)
    diff = max(abs(newV[s] - V[s]) for s in states)
    V = newV
    if it % 5 == 0:
        print(f"iter={it}: V(s0)={V['s0']:.4f}, V(s1)={V['s1']:.4f}, diff={diff:.6f}")
    if diff < 1e-6:
        break

policy_star = {}
for s in states:
    q_left = action_value(s, 'left', V)
    q_right = action_value(s, 'right', V)
    policy_star[s] = 'left' if q_left >= q_right else 'right'

print('optimal V =', {k: round(v, 6) for k, v in V.items()})
print('greedy policy from V =', policy_star)
